# MovieLens 100K - Movie Recommender System

Dataset: [MovieLens 100K](https://grouplens.org/datasets/movielens/100k/) (GroupLens Research). 100,000 ratings, 943 users, 1,682 movies. See README.md for download steps and citation requirement.

**Citation (required by GroupLens' terms of use):** F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM TiiS 5, 4, Article 19.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_ratings, load_items, build_sparse_ratings_matrix, GENRE_COLUMNS
from src.content_based import build_genre_similarity, recommend_similar_movies, recommend_for_user_content_based
from src.collaborative import item_item_similarity, recommend_item_based, fit_svd, recommend_svd, recommend_popular
from src.evaluation import train_test_split_ratings, build_relevant_sets, precision_recall_at_k

## Milestone 1 - Data prep

In [ ]:
ratings = load_ratings()
items = load_items()
print(f"ratings: {ratings.shape}, items: {items.shape}")
print(f"unique users: {ratings['user_id'].nunique()}, unique movies rated: {ratings['item_id'].nunique()}")
ratings.head()

In [ ]:
matrix, user_id_to_row, item_id_to_col, row_to_user_id, col_to_item_id = build_sparse_ratings_matrix(ratings)
density = matrix.nnz / (matrix.shape[0] * matrix.shape[1])
print(f"Sparse ratings matrix: {matrix.shape}, density={density:.4f} ({matrix.nnz} filled cells)")

## Milestone 2 - Content-based filtering (genre similarity)

In [ ]:
genre_sim = build_genre_similarity(items)
print(f"Genre similarity matrix shape: {genre_sim.shape}")

# Pick any movie_id present in your data to sanity-check this.
example_movie_id = int(items['movie_id'].iloc[0])
recommend_similar_movies(example_movie_id, items, genre_sim, k=5)

## Milestone 3 - Collaborative filtering: item-based CF vs. SVD

In [ ]:
item_sim = item_item_similarity(matrix)
print(f"Item-item similarity matrix: {item_sim.shape}")

example_user_row = 0
cf_recs = recommend_item_based(example_user_row, matrix, item_sim, col_to_item_id, k=10)
print("Item-based CF top-10 for user row 0:", cf_recs)

In [ ]:
svd, user_factors = fit_svd(matrix, n_components=20)
svd_recs = recommend_svd(example_user_row, matrix, svd, user_factors, col_to_item_id, k=10)
print("SVD top-10 for user row 0:", svd_recs)
print(f"\nExplained variance ratio (sum): {svd.explained_variance_ratio_.sum():.3f}")

**Comparison note:** _(TODO(you) - do item-based CF and SVD agree on any recommendations for this user? Which would you pick to productionize and why - consider both accuracy and how each behaves as the catalog grows.)_

## Milestone 4 - Hybrid (content-based + collaborative blend)

In [ ]:
def hybrid_recommend(user_id, ratings, items, genre_sim, matrix, item_sim,
                      user_id_to_row, col_to_item_id, k=10, cf_weight=0.6):
    """Simple weighted blend: cf_weight on the CF score, (1-cf_weight)
    on the content-based score, both re-scaled to [0, 1] before blending
    so neither dominates purely due to differing score ranges."""
    content_recs = recommend_for_user_content_based(user_id, ratings, items, genre_sim, k=50)
    if user_id not in user_id_to_row:
        return recommend_popular(ratings, items, k=k)  # cold start
    row = user_id_to_row[user_id]
    cf_recs = recommend_item_based(row, matrix, item_sim, col_to_item_id, k=50)

    cf_scores = {iid: score for iid, score in cf_recs}
    content_scores = dict(zip(content_recs["movie_id"], content_recs["score"])) if not content_recs.empty else {}

    def normalize(d):
        if not d:
            return d
        vals = np.array(list(d.values()))
        lo, hi = vals.min(), vals.max()
        if hi - lo < 1e-9:
            return {k: 0.5 for k in d}
        return {k: (v - lo) / (hi - lo) for k, v in d.items()}

    cf_norm = normalize(cf_scores)
    content_norm = normalize(content_scores)

    all_ids = set(cf_norm) | set(content_norm)
    blended = {
        iid: cf_weight * cf_norm.get(iid, 0) + (1 - cf_weight) * content_norm.get(iid, 0)
        for iid in all_ids
    }
    top_ids = sorted(blended, key=blended.get, reverse=True)[:k]
    return items[items["movie_id"].isin(top_ids)][["movie_id", "movie_title"]]


example_user_id = row_to_user_id[example_user_row]
hybrid_recommend(example_user_id, ratings, items, genre_sim, matrix, item_sim,
                  user_id_to_row, col_to_item_id, k=10)

## Milestone 5 - Cold start

In [ ]:
# A user_id that has never rated anything, e.g. one bigger than any real user_id.
brand_new_user_id = int(ratings["user_id"].max()) + 1000
print("Cold-start recommendation (must not crash or be empty):")
recommend_popular(ratings, items, k=10)

## Milestone 6 - Evaluation: precision@10 / recall@10

In [ ]:
train_ratings, test_ratings = train_test_split_ratings(ratings, test_frac=0.2)
relevant = build_relevant_sets(test_ratings, relevance_threshold=4)

train_matrix, tr_u2r, tr_i2c, tr_r2u, tr_c2i = build_sparse_ratings_matrix(train_ratings)
train_item_sim = item_item_similarity(train_matrix)

recommended = {}
for user_id, row in tr_u2r.items():
    recs = recommend_item_based(row, train_matrix, train_item_sim, tr_c2i, k=10)
    recommended[user_id] = [iid for iid, _ in recs]

precision, recall = precision_recall_at_k(recommended, relevant, k=10)
print(f"Item-based CF: precision@10 = {precision:.4f}, recall@10 = {recall:.4f}")

**Honest interpretation:** _(TODO(you) - precision@10 around 0.05-0.15 is typical for this kind of setup on MovieLens 100K with a simple split; state your actual number and whether it's in a reasonable range, don't just report it without context.)_